sieci dwuwarstwowe w Pythonie - stud.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1S6ha3t69w4MCmNS6knix8zlaI8NbgF6s

## funkcja `init2`

In [ ]:
import numpy as np

In [ ]:
def init2(num_inputs: int, num_hidden: int, num_outputs: int):
  """
  Initializes a two-layer neural network with random weights.

  The function creates a two-layer network and fills its weight matrices
  with random values from the range [-0.1, 0.1].

  Args:
    num_inputs: Number of inputs to the network / Number of inputs for layer 1.
    num_hidden: Number of neurons in layer 1 (the hidden layer).
    num_outputs: Number of neurons in layer 2 / Number of network outputs.

  Returns:
    Tuple[np.ndarray, np.ndarray]:
      weights_layer1: Weight matrix for layer 1 of the network.
      weights_layer2: Weight matrix for layer 2 of the network.
  """
  weights_layer1 = np.random.rand(num_inputs + 1, num_hidden) * 0.2 - 0.1

  weights_layer2 = np.random.rand(num_hidden + 1, num_outputs) * 0.2 - 0.1

  return weights_layer1, weights_layer2

Let's test the `init2` function with an example. Suppose we have:
- `num_inputs = 10` inputs
- `num_hidden = 5` neurons in the first hidden layer
- `num_outputs = 2` neurons in the output layer

In [ ]:
# Example usage:
num_inputs = 10
num_hidden = 5
num_outputs = 2

weights_layer1, weights_layer2 = init2(num_inputs, num_hidden, num_outputs)

print(f"Shape of weights_layer1 (Layer 1 weights): {weights_layer1.shape}")
print(f"First 3 rows of weights_layer1:\n{weights_layer1[:3, :]}")
print("---")
print(f"Shape of weights_layer2 (Layer 2 weights): {weights_layer2.shape}")
print(f"First 3 rows of weights_layer2:\n{weights_layer2[:3, :]}")

# Verify the range of weights
print(f"\nMin value in weights_layer1: {np.min(weights_layer1):.3f}, Max value in weights_layer1: {np.max(weights_layer1):.3f}")
print(f"Min value in weights_layer2: {np.min(weights_layer2):.3f}, Max value in weights_layer2: {np.max(weights_layer2):.3f}")

## funkcja `run2`

In [ ]:
def run2(weights_layer1: np.ndarray, weights_layer2: np.ndarray, input_vector: np.ndarray):
  """
  Simulates the operation of a two-layer neural network.

  Args:
    weights_layer1: Weight matrix for the first layer of the network.
    weights_layer2: Weight matrix for the second layer of the network.
    input_vector: Input vector to the network (signal fed to the input / layer 1).

  Returns:
    Tuple[np.ndarray, np.ndarray]:
      output_layer1: Output vector of layer 1 (useful during training).
      output_layer2: Output vector of layer 2 / network output.
  """
  beta = 5

  layer1_input = np.vstack([-1, input_vector.reshape(-1, 1)])

  net_input_layer1 = weights_layer1.T @ layer1_input

  output_layer1 = 1 / (1 + np.exp(-beta * net_input_layer1))

  layer2_input = np.vstack([-1, output_layer1.reshape(-1, 1)])

  net_input_layer2 = weights_layer2.T @ layer2_input

  output_layer2 = 1 / (1 + np.exp(-beta * net_input_layer2))

  return output_layer1, output_layer2

Now, let's test the `run2` function using the `weights_layer1` and `weights_layer2` generated previously, and a sample input vector.

In [ ]:
# Example usage for run2:

# Create a sample input vector.
# It should have num_inputs elements. We generated weights_layer1 based on num_inputs = 10.
# Ensure it is a column vector for matrix multiplication with weights_layer1.T
input_vector = np.random.rand(num_inputs, 1) * 2 - 1 # Random values between -1 and 1

print(f"Sample input vector (shape: {input_vector.shape}):\n{input_vector[:5].T}...")

# Perform the forward pass
output_layer1, output_layer2 = run2(weights_layer1, weights_layer2, input_vector)

print(f"\nOutput of Layer 1 (shape: {output_layer1.shape}):\n{output_layer1[:5].T}...")
print(f"\nOutput of Layer 2 (shape: {output_layer2.shape}):\n{output_layer2.T}")

# Verify the range of outputs (sigmoid typically outputs between 0 and 1)
print(f"\nMin value in output_layer1: {np.min(output_layer1):.3f}, Max value in output_layer1: {np.max(output_layer1):.3f}")
print(f"Min value in output_layer2: {np.min(output_layer2):.3f}, Max value in output_layer2: {np.max(output_layer2):.3f}")

## funkcja `train2`

In [ ]:
def train2(weights_layer1_before: np.ndarray, weights_layer2_before: np.ndarray, training_inputs: np.ndarray, targets: np.ndarray, num_iterations: int):
  """
  Trains a two-layer neural network using backpropagation.

  Args:
    weights_layer1_before: Initial weight matrix for the first layer.
    weights_layer2_before: Initial weight matrix for the second layer.
    training_inputs: Input patterns (matrix where each column is an input example).
    targets: Target outputs (matrix where each column is a target example).
    num_iterations: Number of training iterations.

  Returns:
    Tuple[np.ndarray, np.ndarray]:
      weights_layer1_after: Trained weight matrix for layer 1.
      weights_layer2_after: Trained weight matrix for layer 2.
  """
  num_examples = training_inputs.shape[1]
  weights_layer1 = weights_layer1_before.copy()
  weights_layer2 = weights_layer2_before.copy()
  learning_rate = 0.1  # learning rate
  beta = 5             # Activation function parameter (from run2)

  for step in range(num_iterations):
    # Randomly select one example
    example_idx = np.random.randint(num_examples) # 0-based indexing in Python
    input_vector = training_inputs[:, example_idx].reshape(-1, 1) # Column vector
    target = targets[:, example_idx].reshape(-1, 1) # Column vector

    # Forward Pass
    output_layer1, output_layer2 = run2(weights_layer1, weights_layer2, input_vector)

    # Build the layer inputs (with bias) explicitly for the weight updates
    layer1_input = np.vstack([-1, input_vector])
    layer2_input = np.vstack([-1, output_layer1])

    # Backpropagation - gradient of the output layer
    error_layer2 = target - output_layer2
    delta_layer2 = beta * error_layer2 * output_layer2 * (1 - output_layer2)

    # Backpropagation - gradient of the hidden layer (skip the bias weight)
    error_layer1 = weights_layer2[1:, :] @ delta_layer2
    delta_layer1 = beta * error_layer1 * output_layer1 * (1 - output_layer1)

    # Calculate weight corrections
    weight_update_layer1 = learning_rate * layer1_input @ delta_layer1.T
    weight_update_layer2 = learning_rate * layer2_input @ delta_layer2.T

    # Apply corrections
    weights_layer1 = weights_layer1 + weight_update_layer1
    weights_layer2 = weights_layer2 + weight_update_layer2

  weights_layer1_after = weights_layer1
  weights_layer2_after = weights_layer2

  return weights_layer1_after, weights_layer2_after

## przykład: XOR

This script demonstrates a two-layer neural network solving the XOR problem. We'll set up the input patterns `training_inputs` and target outputs `targets`, initialize the network, perform a forward pass before training, and then run the training step (`train2`). Finally, we'll perform another forward pass to see the output after training.

In [ ]:
# Input patterns for XOR problem
training_inputs = np.array([
    [0, 0, 1, 1],   # Input 1
    [0, 1, 0, 1]    # Input 2
])

# Desired output for XOR problem
targets = np.array([
    [0, 1, 1, 0]    # Target output
])

print("Input patterns:\n", training_inputs)
print("Target outputs:\n", targets)

### Network Initialization and Pre-Training Forward Pass

We initialize the network with 2 inputs (num_inputs=2), 2 neurons in the first layer (num_hidden=2), and 1 output neuron (num_outputs=1). Then, we perform a forward pass for each input pattern to see the network's output *before* any training.

In [ ]:
# Initialize weights for a small network (num_inputs=2, num_hidden=2, num_outputs=1)
num_inputs = 2
num_hidden = 2
num_outputs = 1

weights_layer1_before, weights_layer2_before = init2(num_inputs, num_hidden, num_outputs)

# Perform forward pass for each input pattern before training
outputs_before = []
for example_idx in range(training_inputs.shape[1]):
    # Select the example_idx-th column as an input vector
    # and reshape it to a column vector for run2
    input_vector = training_inputs[:, example_idx].reshape(-1, 1)
    output_layer1, output_layer2 = run2(weights_layer1_before, weights_layer2_before, input_vector)
    outputs_before.append(output_layer2.flatten()[0])

outputs_before = np.array([outputs_before])

print(f"weights_layer1_before shape: {weights_layer1_before.shape}, weights_layer2_before shape: {weights_layer2_before.shape}")
print("Network output before training:\n", outputs_before)
print("Target output:\n", targets)

### Training the Network

In [ ]:
# Call the training function
num_iterations = 5000
weights_layer1_after, weights_layer2_after = train2(weights_layer1_before, weights_layer2_before, training_inputs, targets, num_iterations)

print("\nWeights after training: ")
print(f"weights_layer1_after shape: {weights_layer1_after.shape}, weights_layer2_after shape: {weights_layer2_after.shape}")

### Post-Training Forward Pass

After the training, we perform another forward pass using the trained weights to see the network's output. If `train2` successfully trained the network, the outputs would be closer to the targets.

In [ ]:
# Perform forward pass for each input pattern after training
outputs_after = []
for example_idx in range(training_inputs.shape[1]):
    input_vector = training_inputs[:, example_idx].reshape(-1, 1)
    output_layer1, output_layer2 = run2(weights_layer1_after, weights_layer2_after, input_vector)
    outputs_after.append(output_layer2.flatten()[0])

outputs_after = np.array([outputs_after])

print("Network output after training:\n", outputs_after)
print("Target output:\n", targets)